# IRWA Final Project. Part 2: Indexing and Evaluation

Course: Information Retrieval and Web Analytics (IRWA)
Group members: Sergi Cases, Julia Garcia, Martí Casas

In this part we build an inverted index over the processed corpus from Part 1, implement a conjunctive (AND) search, rank the results with TF-IDF and cosine similarity, and evaluate the search engine with the usual IR metrics.

The code is in the src folder so that it can be reused in Parts 3 and 4:

- src/preprocessing.py: build_terms, copied from Part 1 without changes, used for the queries
- src/indexing.py: loading the corpus, inverted index, idf, document norms and AND search
- src/ranking.py: TF-IDF + cosine ranking
- src/evaluation.py: evaluation metrics

## 0. Setup

In [1]:
import os

import pandas as pd

from src.preprocessing import build_terms
from src.indexing import load_corpus, build_index, compute_idf, compute_doc_norms, conjunctive_search
from src.ranking import rank_tfidf, search
from src.evaluation import (precision_at_k, recall_at_k, avg_precision_at_k, f1_at_k, reciprocal_rank,
                            ndcg_at_k, mean_avg_precision, mean_reciprocal_rank, evaluate_rankings,
                            evaluate_labelled_csv)

DATA_DIR = "./"
CORPUS_PATH = DATA_DIR + "processed_corpus.json"
VALIDATION_PATH = DATA_DIR + "validation_labels.csv"
TEST_RESULTS_PATH = DATA_DIR + "test_queries_results.csv"

pd.set_option("display.max_colwidth", 80)

## 1. Indexing

### 1.1 Load the processed corpus

We load processed_corpus.json (one product per line). The searchable text is the all_terms field, which already contains the processed terms of the title, description, useful product details and brand.

In [2]:
corpus = load_corpus(CORPUS_PATH)
titles = corpus.set_index("pid")["title"]
print("Documents:", len(corpus))
corpus[["pid", "title", "all_terms"]].head(3)

Documents: 28080


,pid,title,all_terms
0,TKPFCZ9EA7H5FYZH,Solid Women Multicolor Track Pants,"[solid, women, multicolor, track, pant, yorker, trackpant, made, rich, comb,..."
1,TKPFCZ9EJZV2UVRZ,Solid Men Blue Track Pants,"[solid, men, blue, track, pant, yorker, trackpant, made, rich, comb, cotton,..."
2,TKPFCZ9EHFCY5Z4Y,Solid Men Multicolor Track Pants,"[solid, men, multicolor, track, pant, yorker, trackpant, made, rich, comb, c..."


### 1.2 Query preprocessing

Queries must be processed exactly like the documents, otherwise a query word would not match the stems stored in the index. We use the same build_terms function of Part 1.

In [3]:
for q in ["women full sleeve sweatshirt cotton", "men slim jeans blue", "Men's Slim Fit T-Shirt"]:
    print(f"{q:38} -> {build_terms(q)}")

women full sleeve sweatshirt cotton    -> ['women', 'full', 'sleev', 'sweatshirt', 'cotton']
men slim jeans blue                    -> ['men', 'slim', 'jean', 'blue']
Men's Slim Fit T-Shirt                 -> ['men', 'slim', 'fit', 'tshirt']


### 1.3 Build the inverted index

For every term we store a posting list with the documents that contain it and the term frequency in each one (term -> {pid: tf}). With the index we also compute:

- idf of every term: log(N / df), where df is the number of documents with the term
- the norm of the tf-idf vector of every document, needed later for the cosine similarity

In [4]:
%%time
index = build_index(corpus, field="all_terms")
idf = compute_idf(index, len(corpus))
doc_norms = compute_doc_norms(index, idf)
print("Vocabulary size:", len(index))

Vocabulary size: 5335
CPU times: total: 844 ms
Wall time: 868 ms


In [5]:
# Example: posting list of a term
term = build_terms("sweatshirt")[0]
print(f"Term '{term}': df = {len(index[term])}, idf = {idf[term]:.3f}")
print("First postings (pid: tf):", list(index[term].items())[:5])

Term 'sweatshirt': df = 1417, idf = 2.987
First postings (pid: tf): [('SWSFYD33P2XCBWFG', 1), ('SWSFYD3KMGTBZEJZ', 1), ('SWSFYD38Y2MAT9FQ', 1), ('SWSFVEV2YFEJKUCK', 2), ('SWSFVEV2BWZJDJ46', 1)]


### 1.4 Conjunctive (AND) search

A document is returned only if it contains all the query terms. The search intersects the posting lists of the query terms, starting with the shortest one. If a term is not in the vocabulary, no document is returned.

In [6]:
for q in ["women full sleeve sweatshirt cotton", "men slim jeans blue", "red wool scarf", "unicornword shirt"]:
    print(f"{q:38} -> {len(conjunctive_search(q, index))} documents")

women full sleeve sweatshirt cotton    -> 549 documents
men slim jeans blue                    -> 220 documents
red wool scarf                         -> 0 documents
unicornword shirt                      -> 0 documents


### 1.5 TF-IDF ranking

The documents returned by the AND search are ranked with the cosine similarity between the tf-idf vector of the query and the tf-idf vector of the document (weight = tf * idf). The document norm uses all its terms, so a product whose text is mostly about the query gets a higher score than a long text that only mentions the query terms. Ties are broken by pid so the order is always the same.

In [7]:
def show_results(query, top_k=10):
    """Prints the top_k ranked results of a query with their titles."""
    ranking = search(query, index, idf, doc_norms, top_k=top_k)
    table = pd.DataFrame(ranking, columns=["pid", "score"])
    table["title"] = table["pid"].map(titles)
    table.index = range(1, len(table) + 1)
    print(f"Query: '{query}' ({len(conjunctive_search(query, index))} documents match all terms)")
    display(table.round(3))


show_results("women full sleeve sweatshirt cotton")
show_results("men slim jeans blue")

Query: 'women full sleeve sweatshirt cotton' (549 documents match all terms)


,pid,score,title
1,SWSFZVTTQCB4SJ7F,0.847,Full Sleeve Solid Women Sweatshirt
2,SWSFZVTNGM4HG8BC,0.832,Full Sleeve Printed Women Sweatshirt
3,SWSFQGS456JAZCQB,0.760,Full Sleeve Printed Women Sweatshirt
4,SWSEXAHDUWHFZDHC,0.759,Full Sleeve Printed Women Sweatshirt
5,SWSEXAHDQQCKCAEA,0.754,Full Sleeve Printed Women Sweatshirt
6,SWSFV5JNFW5HZQWY,0.753,Full Sleeve Printed Women Sweatshirt
7,SWSFMTNHEM5SNHVZ,0.745,Full Sleeve Solid Women Sweatshirt
8,SWSFYRKYAHH4HHSM,0.735,Full Sleeve Printed Women Sweatshirt
9,SWSFYTYMNTBNARUN,0.729,Full Sleeve Solid Women Sweatshirt
10,SWSFY37UKWPG4AZ2,0.728,Full Sleeve Solid Women Sweatshirt


Query: 'men slim jeans blue' (220 documents match all terms)


,pid,score,title
1,JEAFRAQXEKGUPNUN,0.558,Slim Men Blue Jeans
2,JEAFEKQZ4C2Z6GCX,0.544,Slim Men Blue Jeans
3,JEAFEKQZQPYHHVGS,0.544,Slim Men Blue Jeans
4,JEAFS57EYDGY4EJM,0.540,Slim Men Blue Jeans
5,JEAFSKYHSYMTKEJD,0.530,Slim Men Blue Jeans
6,JEAFTGSGTYKZGAEZ,0.530,Slim Men Blue Jeans
7,JEAFM3Y9ZNMVG6ZM,0.528,Slim Men Blue Jeans
8,JEAFSKYHZHSZZC9S,0.525,Slim Men Blue Jeans
9,JEAFJA4M7SMZN2XJ,0.517,Slim Men Black Jeans
10,JEAFKWR3FX9NH2V6,0.516,Slim Men Blue Jeans


## 2. Evaluation

### 2.1 Metrics

All the metrics are in src/evaluation.py. A ranking is represented by the list of its labels in order (1 = relevant, 0 = not relevant). R is the total number of relevant documents for the query.

- P@K: relevant documents in the top K / K
- R@K: relevant documents in the top K / R
- AP@K: sum of P@i at every position i <= K with a relevant document, divided by R
- F1@K: 2 * P@K * R@K / (P@K + R@K)
- RR: 1 / position of the first relevant document. MRR is its mean over the queries
- MAP: mean of AP@K over the queries
- NDCG@K: DCG@K / ideal DCG@K, with DCG@K = sum of (2^rel - 1) / log2(i + 1)

### 2.2 Small example to check the metrics

Query A has the ranking [1, 0, 1, 0, 0, 1] and R = 4 relevant documents (one of them is not retrieved). Query B has [0, 0, 1, 1, 0] and R = 2. With K = 5, by hand:

- A: P@5 = 2/5 = 0.4, R@5 = 2/4 = 0.5, AP@5 = (1/1 + 2/3) / 4 = 0.417, F1@5 = 2 * 0.4 * 0.5 / 0.9 = 0.444, RR = 1, NDCG@5 = (1 + 1/log2(4)) / (1 + 1/log2(3) + 1/log2(4) + 1/log2(5)) = 1.5 / 2.562 = 0.586
- B: P@5 = 0.4, R@5 = 1, AP@5 = (1/3 + 2/4) / 2 = 0.417, F1@5 = 0.571, RR = 1/3
- MAP = (0.417 + 0.417) / 2 = 0.417 and MRR = (1 + 1/3) / 2 = 0.667

In [8]:
a, b = [1, 0, 1, 0, 0, 1], [0, 0, 1, 1, 0]

checks = {
    "P@5 (A)": (precision_at_k(a, 5), 0.4),
    "R@5 (A)": (recall_at_k(a, 5, n_relevant=4), 0.5),
    "AP@5 (A)": (avg_precision_at_k(a, 5, n_relevant=4), 0.417),
    "F1@5 (A)": (f1_at_k(a, 5, n_relevant=4), 0.444),
    "RR (A)": (reciprocal_rank(a), 1.0),
    "NDCG@5 (A)": (ndcg_at_k(a, 5, n_relevant=4), 0.586),
    "RR (B)": (reciprocal_rank(b), 0.333),
    "MAP@5": (mean_avg_precision([a, b], 5, n_relevants=[4, 2]), 0.417),
    "MRR": (mean_reciprocal_rank([a, b]), 0.667),
}
for name, (value, expected) in checks.items():
    assert round(value, 3) == expected, name
    print(f"{name:12} computed = {value:.3f}   expected = {expected}")

evaluate_rankings({"A": a, "B": b}, k=5, n_relevants={"A": 4, "B": 2})

P@5 (A)      computed = 0.400   expected = 0.4
R@5 (A)      computed = 0.500   expected = 0.5
AP@5 (A)     computed = 0.417   expected = 0.417
F1@5 (A)     computed = 0.444   expected = 0.444
RR (A)       computed = 1.000   expected = 1.0
NDCG@5 (A)   computed = 0.586   expected = 0.586
RR (B)       computed = 0.333   expected = 0.333
MAP@5        computed = 0.417   expected = 0.417
MRR          computed = 0.667   expected = 0.667


,P@K,R@K,AP@K,F1@K,RR,NDCG@K
A,0.4,0.50,0.417,0.444,1.000,0.586
B,0.4,1.00,0.417,0.571,0.333,0.571
mean (MAP / MRR),0.4,0.75,0.417,0.508,0.667,0.578


### 2.3 Validation queries

validation_labels.csv contains 20 labelled results for each of the two validation queries. We consider it as the output of a search engine with its labels, so we evaluate the ranking in the same order as it appears in the file. R is the number of documents labelled as relevant for each query (13 for query 1 and 10 for query 2). The last row of each table is the mean over the two queries, so the AP@K column gives MAP and the RR column gives MRR.

In [9]:
VALIDATION_QUERIES = {1: "women full sleeve sweatshirt cotton", 2: "men slim jeans blue"}
K_VALUES = [5, 10, 20]

validation = pd.read_csv(VALIDATION_PATH, dtype={"pid": str})
n_relevants = validation.groupby("query_id")["labels"].sum().to_dict()
print("Relevant documents per query:", n_relevants)

# Ranking in the order of the file
file_rankings = {qid: g["labels"].tolist() for qid, g in validation.groupby("query_id", sort=False)}


def metrics_table(rankings):
    """Metrics of the rankings for every K in K_VALUES, in one table."""
    tables = {f"K={k}": evaluate_rankings(rankings, k, n_relevants) for k in K_VALUES}
    return pd.concat(tables)


metrics_table(file_rankings)

Relevant documents per query: {1: 13, 2: 10}


P@K    R@K   AP@K   F1@K   RR  NDCG@K
K=5  1                 0.600  0.231  0.186  0.333  1.0   0.655
     2                 0.600  0.300  0.210  0.400  1.0   0.616
     mean (MAP / MRR)  0.600  0.265  0.198  0.367  1.0   0.636
K=10 1                 0.700  0.538  0.390  0.609  1.0   0.703
     2                 0.600  0.600  0.408  0.600  1.0   0.615
     mean (MAP / MRR)  0.650  0.569  0.399  0.604  1.0   0.659
K=20 1                 0.650  1.000  0.694  0.788  1.0   0.873
     2                 0.500  1.000  0.627  0.667  1.0   0.833
     mean (MAP / MRR)  0.575  1.000  0.660  0.727  1.0   0.853

### 2.4 Extra: our TF-IDF ranking of the labelled documents

As an extra, we rank the 40 labelled documents with our TF-IDF score for their query (only these documents, without the AND filter, so that all 20 are ranked) and evaluate this new order with the same labels.

In [10]:
tfidf_rankings = {}
for qid, query in VALIDATION_QUERIES.items():
    labelled = validation[validation["query_id"] == qid]
    labels_by_pid = dict(zip(labelled["pid"], labelled["labels"]))
    ranking = rank_tfidf(query, labelled["pid"], index, idf, doc_norms)
    tfidf_rankings[qid] = [labels_by_pid[pid] for pid, _ in ranking]
    n_and = len(set(labelled["pid"]) & conjunctive_search(query, index))
    print(f"Query {qid}: {n_and}/20 labelled documents contain all terms. Labels in TF-IDF order: {tfidf_rankings[qid]}")

metrics_table(tfidf_rankings)

Query 1: 20/20 labelled documents contain all terms. Labels in TF-IDF order: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1]
Query 2: 8/20 labelled documents contain all terms. Labels in TF-IDF order: [1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0]


P@K    R@K   AP@K   F1@K   RR  NDCG@K
K=5  1                 1.000  0.385  0.385  0.556  1.0   1.000
     2                 0.800  0.400  0.380  0.533  1.0   0.854
     mean (MAP / MRR)  0.900  0.392  0.382  0.544  1.0   0.927
K=10 1                 1.000  0.769  0.769  0.870  1.0   1.000
     2                 0.600  0.600  0.511  0.600  1.0   0.691
     mean (MAP / MRR)  0.800  0.685  0.640  0.735  1.0   0.846
K=20 1                 0.650  1.000  0.967  0.788  1.0   0.992
     2                 0.500  1.000  0.762  0.667  1.0   0.920
     mean (MAP / MRR)  0.575  1.000  0.865  0.727  1.0   0.956

In [11]:
# Side by side comparison at K = 10
comparison = pd.concat({"file order": evaluate_rankings(file_rankings, 10, n_relevants),
                        "TF-IDF order": evaluate_rankings(tfidf_rankings, 10, n_relevants)})
comparison

P@K    R@K   AP@K   F1@K   RR  NDCG@K
file order   1                 0.70  0.538  0.390  0.609  1.0   0.703
             2                 0.60  0.600  0.408  0.600  1.0   0.615
             mean (MAP / MRR)  0.65  0.569  0.399  0.604  1.0   0.659
TF-IDF order 1                 1.00  0.769  0.769  0.870  1.0   1.000
             2                 0.60  0.600  0.511  0.600  1.0   0.691
             mean (MAP / MRR)  0.80  0.685  0.640  0.735  1.0   0.846